# TiniMind Training v3
LLM Bahasa Indonesia dari nol — pretrain + SFT

**Struktur folder output (hanya 2):**
```
TiniMind_Prototype/output/
├── pretrain/   ← checkpoint pretrain
└── sft/        ← checkpoint SFT
```

In [ ]:
# Cell 1 — Mount Drive & Setup
from google.colab import drive
drive.mount('/content/drive')

import sys, os, glob

BASE         = '/content/drive/MyDrive/TiniMind_Prototype'
DATA_DIR     = f'{BASE}/data/mc4_indo'          # chunk_*.bin dari Wikipedia + CulturaX
PRETRAIN_DIR = f'{BASE}/output/pretrain'         # checkpoint pretrain
SFT_DIR      = f'{BASE}/output/sft'              # checkpoint SFT
SFT_DATA_DIR = f'{BASE}/data/sft'               # chunk SFT yang sudah ditokenisasi
SFT_JSONL    = f'{BASE}/data/sft_200_final.jsonl'
TOK_PATH     = f'{BASE}/tokenizer/indo_bpe_32k.model'

# Hanya buat 2 folder output yang diperlukan
os.makedirs(PRETRAIN_DIR, exist_ok=True)
os.makedirs(SFT_DIR,      exist_ok=True)

sys.path.insert(0, BASE)

import torch
print(f'Device : {"cuda" if torch.cuda.is_available() else "cpu"}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# Cell 2 — Install dependencies
%pip install -q sentencepiece datasets

In [ ]:
# Cell 3 — Cek data & checkpoint pretrain terakhir
chunks    = sorted(glob.glob(f'{DATA_DIR}/chunk_*.bin'))
total_tok = sum(os.path.getsize(f)//2 for f in chunks)
print(f'Chunks    : {len(chunks)} file')
print(f'Total tok : {total_tok/1e9:.2f}B')

# Checkpoint pretrain — selalu cari di output/pretrain/
pretrain_ckpts = sorted(glob.glob(f'{PRETRAIN_DIR}/step_*.pt'))
if pretrain_ckpts:
    RESUME = pretrain_ckpts[-1]
    print(f'Resume    : {os.path.basename(RESUME)}')
else:
    RESUME = None
    print('Resume    : tidak ada — training dari awal')

if not chunks:
    print('\n⚠️  Belum ada data! Jalankan Cell 4 dulu.')

In [ ]:
# Cell 4 — Stream & tokenize CulturaX → chunk_*.bin
# SKIP kalau data/mc4_indo sudah punya cukup chunk (3B token = ~300 file)

import sentencepiece as spm
import numpy as np
from datasets import load_dataset

TARGET_TOKENS = 3_000_000_000
CHUNK_SIZE    = 10_000_000

sp = spm.SentencePieceProcessor()
sp.Load(TOK_PATH)
print(f'Tokenizer vocab: {sp.GetPieceSize()}')

os.makedirs(DATA_DIR, exist_ok=True)
existing   = sorted(glob.glob(f'{DATA_DIR}/chunk_*.bin'))
tokens_done = sum(os.path.getsize(f)//2 for f in existing)
print(f'Token sudah ada: {tokens_done/1e9:.2f}B / {TARGET_TOKENS/1e9:.1f}B')

if tokens_done >= TARGET_TOKENS:
    print('Target sudah tercapai, skip.')
else:
    chunk_idx = len(existing)
    buf = []
    ds  = load_dataset('uonlp/CulturaX', 'id', split='train', streaming=True, trust_remote_code=True)

    for doc in ds:
        toks = sp.Encode(doc['text'])
        buf.extend(toks)
        tokens_done += len(toks)

        while len(buf) >= CHUNK_SIZE:
            path = f'{DATA_DIR}/chunk_{chunk_idx:04d}.bin'
            np.array(buf[:CHUNK_SIZE], dtype=np.uint16).tofile(path)
            print(f'Saved chunk_{chunk_idx:04d}.bin | total: {tokens_done/1e9:.2f}B')
            buf = buf[CHUNK_SIZE:]
            chunk_idx += 1

        if tokens_done >= TARGET_TOKENS:
            print('Target tercapai!')
            break

    if buf:
        path = f'{DATA_DIR}/chunk_{chunk_idx:04d}.bin'
        np.array(buf, dtype=np.uint16).tofile(path)
        print(f'Final: {os.path.basename(path)}')

In [ ]:
# Cell 5 — Pretrain
# Checkpoint disimpan di: output/pretrain/step_XXXXXXX_loss_X.XXXX.pt
# --config prod_300m = 24L x 1024H x 16Q/4KV x vocab=32000 (~303M params)
# --dtype fp16 karena T4 tidak support bf16

RESUME_ARG = f'--resume {RESUME}' if RESUME else ''

!python {BASE}/train.py \
    --config prod_300m \
    --data-dir {DATA_DIR} \
    --output-dir {PRETRAIN_DIR} \
    --dtype fp16 \
    --max-steps 20000 \
    --lr 3e-4 \
    --batch-size 4 \
    --grad-accum 8 \
    --seq-len 1024 \
    --log-every 100 \
    --save-every 1000 \
    --eval-every 1000 \
    {RESUME_ARG}

In [ ]:
# Cell 6 — Tokenize SFT data
# Format input: JSONL dengan {"turns": [["user", "..."], ["assistant", "..."]]}
# Output: data/sft/chunk_0000.bin (satu file, semua conversation)

import sentencepiece as spm, json, numpy as np

os.makedirs(SFT_DATA_DIR, exist_ok=True)

sp = spm.SentencePieceProcessor()
sp.Load(TOK_PATH)

# Parse mixed format (compact + pretty-printed)
raw      = open(SFT_JSONL).read()
decoder  = json.JSONDecoder()
entries, i = [], 0
while i < len(raw):
    while i < len(raw) and raw[i] in ' \t\n\r': i += 1
    if i >= len(raw): break
    try:
        obj, end = decoder.raw_decode(raw, i)
        entries.append(obj)
        i = end
    except json.JSONDecodeError:
        i = raw.find('\n', i) + 1 if raw.find('\n', i) != -1 else len(raw)

print(f'SFT entries: {len(entries)}')

all_tokens = []
for conv in entries:
    text = ''
    for role, content in conv['turns']:
        if role == 'user':
            text += f'<penggunna>{content}</penggunna>'
        else:
            text += f'<asisten>{content}</asisten>'
    all_tokens.extend(sp.Encode(text))

out_path = f'{SFT_DATA_DIR}/chunk_0000.bin'
np.array(all_tokens, dtype=np.uint16).tofile(out_path)
print(f'SFT tokens : {len(all_tokens):,}')
print(f'Saved      : {out_path}')

In [ ]:
# Cell 7 — SFT
# Checkpoint disimpan di: output/sft/step_XXXXXXX_loss_X.XXXX.pt
# Mulai dari checkpoint pretrain terbaik

pretrain_ckpts = sorted(glob.glob(f'{PRETRAIN_DIR}/step_*.pt'))
if not pretrain_ckpts:
    raise RuntimeError('Tidak ada checkpoint pretrain! Jalankan Cell 5 dulu.')

BEST_PRETRAIN = pretrain_ckpts[-1]
print(f'Base model: {os.path.basename(BEST_PRETRAIN)}')

# Cek resume SFT
sft_ckpts  = sorted(glob.glob(f'{SFT_DIR}/step_*.pt'))
SFT_RESUME = f'--resume {sft_ckpts[-1]}' if sft_ckpts else f'--resume {BEST_PRETRAIN}'

!python {BASE}/train.py \
    --config prod_300m \
    --data-dir {SFT_DATA_DIR} \
    --output-dir {SFT_DIR} \
    --dtype fp16 \
    --max-steps 500 \
    --lr 1e-5 \
    --batch-size 2 \
    --grad-accum 4 \
    --seq-len 1024 \
    --log-every 50 \
    --save-every 100 \
    --eval-every 100 \
    --val-chunks 0 \
    {SFT_RESUME}

In [ ]:
# Cell 8 — Inference test
import torch
sys.path.insert(0, BASE)
from model_v2 import TiniMind
from config  import ModelConfig
import sentencepiece as spm

# Cari checkpoint terbaik: SFT dulu, fallback ke pretrain
sft_ckpts      = sorted(glob.glob(f'{SFT_DIR}/step_*.pt'))
pretrain_ckpts = sorted(glob.glob(f'{PRETRAIN_DIR}/step_*.pt'))
ckpt_path      = (sft_ckpts or pretrain_ckpts)[-1]
print(f'Load: {os.path.basename(ckpt_path)}')

ckpt   = torch.load(ckpt_path, map_location='cpu', weights_only=False)
cfg    = ckpt.get('config', ModelConfig(
    num_layers=24, hidden_size=1024, num_heads=16, num_kv_heads=4,
    vocab_size=32000, max_seq_len=2048
))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = TiniMind(cfg).to(device)
model.load_state_dict(ckpt['model'])
model.eval()
print(f'Params: {model.num_params()/1e6:.1f}M')

sp = spm.SentencePieceProcessor()
sp.Load(TOK_PATH)

@torch.no_grad()
def generate(prompt, max_new=200, temp=0.8, top_k=50):
    ids      = torch.tensor([sp.Encode(prompt)], dtype=torch.long).to(device)
    past_kvs = None
    for _ in range(max_new):
        inp     = ids if past_kvs is None else ids[:, -1:]
        offset  = 0   if past_kvs is None else ids.shape[1] - 1
        logits, _, past_kvs = model(inp, use_kv_cache=True, past_kvs=past_kvs, offset=offset)
        logits  = logits[:, -1, :] / temp
        if top_k:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, -1:]] = -float('inf')
        nxt = torch.multinomial(torch.softmax(logits, -1), 1)
        ids = torch.cat([ids, nxt], dim=1)
    return sp.Decode(ids[0].tolist())

print(generate('<penggunna>Apa itu kecerdasan buatan?</penggunna><asisten>'))

In [ ]:
# Cell 9 — Quantize INT8
sft_ckpts  = sorted(glob.glob(f'{SFT_DIR}/step_*.pt'))
ckpt_path  = sft_ckpts[-1] if sft_ckpts else sorted(glob.glob(f'{PRETRAIN_DIR}/step_*.pt'))[-1]
quant_out  = f'{BASE}/output/tinimind_int8.pt'

!python {BASE}/quantize.py \
    --checkpoint {ckpt_path} \
    --output {quant_out} \
    --mode dynamic